In [81]:
import pickle
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertModel
from torch.optim import AdamW

with open("data/sarcasm.pkl", "rb") as f:
    data = pickle.load(f)

with open("data/sarcasm_data.json", "r", encoding="utf-8") as f:
    meta = json.load(f)

def build_samples(split):
    split_data = data[split]

    texts = split_data["text"]
    audios = split_data["audio"]
    visions = split_data["vision"]
    ids = split_data["id"]

    samples = []

    for i in range(len(ids)):
        sample_id = ids[i]

        if isinstance(sample_id, bytes):
            sample_id = sample_id.decode()

        # 取 json 信息
        info = meta[sample_id]

        # 拼接 context
        context_list = info["context"]
        context_str = " ".join(context_list)

        sample = [
            info["utterance"],
            context_str,
            audios[i],     #(50, 81)
            visions[i],    #(50, 371)
            int(info['sarcasm'])
        ]

        samples.append(sample)

    return samples

# 3. build
train_samples = build_samples("train")
valid_samples = build_samples("valid")
train_valid_samples = train_samples + valid_samples
test_samples  = build_samples("test")

# Experiment 1

In [107]:
class SarcasmContextDataset(Dataset):
    def __init__(self, sample_list):
        self.context_list = [x[1] for x in sample_list]
        self.label_list = [x[4] for x in sample_list]

    def __len__(self):
        return len(self.context_list)

    def __getitem__(self, idx):
        return self.context_list[idx], self.label_list[idx]
    
class ContextCollator:
    def __init__(self, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, batch):
        context_batch = [x[0] for x in batch]
        label_batch = torch.tensor([x[1] for x in batch], dtype=torch.long)

        tokenized = self.tokenizer(
            context_batch,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": tokenized["input_ids"],
            "attention_mask": tokenized["attention_mask"],
            "labels": label_batch
        }

class BertMLPClassifier(nn.Module):
    def __init__(self, pretrained_name="bert-base-uncased", hidden_dim=256, dropout=0.2, ):
        super().__init__()
        self.bert = BertModel.from_pretrained(pretrained_name)
        self.mlp = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)
        )
        for param in self.bert.parameters():
            param.requires_grad = True

    def forward(self, input_ids, attention_mask):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embed = bert_out.last_hidden_state[:, 0, :]   # [CLS]
        logits = self.mlp(cls_embed)
        return logits

In [108]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_dataset = SarcasmContextDataset(train_valid_samples)
test_dataset = SarcasmContextDataset(test_samples)

collator = ContextCollator(tokenizer, max_length=128)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collator
)

In [109]:
model = BertMLPClassifier(
    pretrained_name="bert-base-uncased",
    hidden_dim=256,
    dropout=0.5
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-6)
criterion = nn.CrossEntropyLoss()

def run_one_epoch(model, data_loader, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for batch_data in data_loader:
        input_ids = batch_data["input_ids"].to(device)
        attention_mask = batch_data["attention_mask"].to(device)
        labels = batch_data["labels"].to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

        preds = torch.argmax(logits, dim=1)
        total_loss += loss.item() * labels.size(0)
        total_correct += (preds == labels).sum().item()
        total_count += labels.size(0)

    avg_loss = total_loss / total_count
    avg_acc = total_correct / total_count
    return avg_loss, avg_acc

num_epochs = 10

for epoch_idx in range(1, num_epochs + 1):
    train_loss, train_acc = run_one_epoch(model, train_loader, optimizer=optimizer)
    test_loss, test_acc = run_one_epoch(model, test_loader, optimizer=None)

    print(
        f"Epoch {epoch_idx:02d} | "
        f"train loss: {train_loss:.4f} | train acc: {train_acc:.4f} | "
        f"test loss: {test_loss:.4f} | test acc: {test_acc:.4f}"
    )

Epoch 01 | train loss: 0.6935 | train acc: 0.5199 | test loss: 0.6935 | test acc: 0.4710
Epoch 02 | train loss: 0.6845 | train acc: 0.5580 | test loss: 0.6813 | test acc: 0.6087
Epoch 03 | train loss: 0.6789 | train acc: 0.5670 | test loss: 0.6723 | test acc: 0.6667
Epoch 04 | train loss: 0.6642 | train acc: 0.6341 | test loss: 0.6616 | test acc: 0.7029
Epoch 05 | train loss: 0.6591 | train acc: 0.6196 | test loss: 0.6574 | test acc: 0.6957
Epoch 06 | train loss: 0.6470 | train acc: 0.6467 | test loss: 0.6495 | test acc: 0.7029
Epoch 07 | train loss: 0.6364 | train acc: 0.6866 | test loss: 0.6366 | test acc: 0.7246
Epoch 08 | train loss: 0.6229 | train acc: 0.6866 | test loss: 0.6258 | test acc: 0.7101
Epoch 09 | train loss: 0.6040 | train acc: 0.7391 | test loss: 0.6174 | test acc: 0.7246
Epoch 10 | train loss: 0.5949 | train acc: 0.7301 | test loss: 0.6129 | test acc: 0.7029


## Expreiment 2

#### BERT(Context + Utterance) + MLP

In [92]:
class SarcasmTextDataset(Dataset):
    def __init__(self, sample_list):
        self.samples = sample_list

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        one_sample = self.samples[idx]
        utterance_text = one_sample[0]
        context_text = one_sample[1]
        label_value = one_sample[4]
        return context_text, utterance_text, label_value
    
class TextCollator:
    def __init__(self, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.tokenizer.truncation_side = "left"

    def __call__(self, batch):
        context_batch = [item[0] for item in batch]
        utterance_batch = [item[1] for item in batch]
        label_batch = torch.tensor([item[2] for item in batch], dtype=torch.long)

        merged_text_batch = []
        for context_text, utterance_text in zip(context_batch, utterance_batch):
            merged_text = context_text + " " + utterance_text
            merged_text_batch.append(merged_text)

        tokenized = self.tokenizer(
            merged_text_batch,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": tokenized["input_ids"],
            "attention_mask": tokenized["attention_mask"],
            "labels": label_batch
        }
    
class BertMLPClassifier(nn.Module):
    def __init__(self, pretrained_name="bert-base-uncased", hidden_dim=256, dropout=0.2):
        super().__init__()
        self.bert = BertModel.from_pretrained(pretrained_name)
        self.classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, input_ids, attention_mask):
        bert_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_embedding = bert_output.last_hidden_state[:, 0, :]
        logits = self.classifier(cls_embedding)
        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_dataset = SarcasmTextDataset(train_valid_samples)
test_dataset = SarcasmTextDataset(test_samples)

collator = TextCollator(tokenizer=tokenizer, max_length=128)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collator 
)

model = BertMLPClassifier(
    pretrained_name="bert-base-uncased",
    hidden_dim=256,
    dropout=0.5
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-6)


# =========================
# 6. One epoch
# =========================
def run_one_epoch(model, data_loader, criterion, device, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss_value = 0.0
    total_correct_num = 0
    total_sample_num = 0

    for batch_data in data_loader:
        input_ids = batch_data["input_ids"].to(device)
        attention_mask = batch_data["attention_mask"].to(device)
        labels = batch_data["labels"].to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

        pred_labels = torch.argmax(logits, dim=1)

        batch_size = labels.size(0)
        total_loss_value += loss.item() * batch_size
        total_correct_num += (pred_labels == labels).sum().item()
        total_sample_num += batch_size

    avg_loss_value = total_loss_value / total_sample_num
    avg_acc_value = total_correct_num / total_sample_num
    return avg_loss_value, avg_acc_value


# 7. Training loop
num_epochs = 10

print("Experiment 2: BERT(context + utterance) + MLP")

for epoch_num in range(1, num_epochs + 1):
    train_loss, train_acc = run_one_epoch(
        model=model,
        data_loader=train_loader,
        criterion=criterion,
        device=device,
        optimizer=optimizer
    )

    test_loss, test_acc = run_one_epoch(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        optimizer=None
    )

    print(
        f"Epoch {epoch_num:02d} | "
        f"train loss: {train_loss:.4f} | "
        f"train acc: {train_acc:.4f} | "
        f"test loss: {test_loss:.4f} | "
        f"test acc: {test_acc:.4f}"
    )

Experiment 2: BERT(context + utterance) + MLP
Epoch 01 | train loss: 0.6955 | train acc: 0.4801 | test loss: 0.6950 | test acc: 0.4710
Epoch 02 | train loss: 0.6878 | train acc: 0.5634 | test loss: 0.6915 | test acc: 0.4783
Epoch 03 | train loss: 0.6694 | train acc: 0.6069 | test loss: 0.6818 | test acc: 0.5000
Epoch 04 | train loss: 0.6499 | train acc: 0.6232 | test loss: 0.6665 | test acc: 0.6014
Epoch 05 | train loss: 0.6319 | train acc: 0.6884 | test loss: 0.6475 | test acc: 0.6377
Epoch 06 | train loss: 0.5961 | train acc: 0.7083 | test loss: 0.6262 | test acc: 0.6739
Epoch 07 | train loss: 0.5644 | train acc: 0.7246 | test loss: 0.6108 | test acc: 0.6884
Epoch 08 | train loss: 0.5300 | train acc: 0.7591 | test loss: 0.5978 | test acc: 0.6739
Epoch 09 | train loss: 0.4945 | train acc: 0.7862 | test loss: 0.5969 | test acc: 0.6667
Epoch 10 | train loss: 0.4467 | train acc: 0.8170 | test loss: 0.5952 | test acc: 0.6812


# 3

In [ ]:
class SarcasmDualTextDataset(Dataset):
    def __init__(self, sample_list):
        self.samples = sample_list

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        one_sample = self.samples[idx]
        utterance_text = one_sample[0]
        context_text = one_sample[1]
        label_value = one_sample[4]
        return context_text, utterance_text, label_value


# context and utterance are tokenized separately
class DualTextCollator:
    def __init__(self, tokenizer, max_length_context=128, max_length_utterance=64):
        self.tokenizer = tokenizer
        self.max_length_context = max_length_context
        self.max_length_utterance = max_length_utterance

    def __call__(self, batch):
        context_batch = [item[0] for item in batch]
        utterance_batch = [item[1] for item in batch]
        label_batch = torch.tensor([item[2] for item in batch], dtype=torch.long)

        context_tokenized = self.tokenizer(
            context_batch,
            padding=True,
            truncation=True,
            max_length=self.max_length_context,
            return_tensors="pt"
        )

        utterance_tokenized = self.tokenizer(
            utterance_batch,
            padding=True,
            truncation=True,
            max_length=self.max_length_utterance,
            return_tensors="pt"
        )

        return {
            "context_input_ids": context_tokenized["input_ids"],
            "context_attention_mask": context_tokenized["attention_mask"],
            "utterance_input_ids": utterance_tokenized["input_ids"],
            "utterance_attention_mask": utterance_tokenized["attention_mask"],
            "labels": label_batch
        }


# 3. Model
# concat(BERT(context), BERT(utterance)) + MLP
# shared BERT encoder
class BertConcatMLPClassifier(nn.Module):
    def __init__(self, pretrained_name="bert-base-uncased", hidden_dim=256, dropout=0.2):
        super().__init__()
        self.bert = BertModel.from_pretrained(pretrained_name)
        bert_hidden_size = self.bert.config.hidden_size

        self.classifier = nn.Sequential(
            nn.Linear(bert_hidden_size * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)
        )

    def forward(
        self,
        context_input_ids,
        context_attention_mask,
        utterance_input_ids,
        utterance_attention_mask
    ):
        context_output = self.bert(
            input_ids=context_input_ids,
            attention_mask=context_attention_mask
        )
        utterance_output = self.bert(
            input_ids=utterance_input_ids,
            attention_mask=utterance_attention_mask
        )

        context_cls = context_output.last_hidden_state[:, 0, :]
        utterance_cls = utterance_output.last_hidden_state[:, 0, :]

        fused_feature = torch.cat([context_cls, utterance_cls], dim=1)
        logits = self.classifier(fused_feature)
        return logits


# 4. Prepare data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_dataset = SarcasmDualTextDataset(train_valid_samples)
test_dataset = SarcasmDualTextDataset(test_samples)

collator = DualTextCollator(
    tokenizer=tokenizer,
    max_length_context=128,
    max_length_utterance=64
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collator
)


# 5. Model / loss / optimizer
model = BertConcatMLPClassifier(
    pretrained_name="bert-base-uncased",
    hidden_dim=256,
    dropout=0.2
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-6)


# 6. One epoch
def run_one_epoch(model, data_loader, criterion, device, optimizer=None):
    is_train = optimizer is not None

    if is_train:
        model.train()
    else:
        model.eval()

    total_loss_value = 0.0
    total_correct_num = 0
    total_sample_num = 0

    for batch_data in data_loader:
        context_input_ids = batch_data["context_input_ids"].to(device)
        context_attention_mask = batch_data["context_attention_mask"].to(device)
        utterance_input_ids = batch_data["utterance_input_ids"].to(device)
        utterance_attention_mask = batch_data["utterance_attention_mask"].to(device)
        labels = batch_data["labels"].to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            logits = model(
                context_input_ids=context_input_ids,
                context_attention_mask=context_attention_mask,
                utterance_input_ids=utterance_input_ids,
                utterance_attention_mask=utterance_attention_mask
            )
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                optimizer.step()

        pred_labels = torch.argmax(logits, dim=1)

        batch_size = labels.size(0)
        total_loss_value += loss.item() * batch_size
        total_correct_num += (pred_labels == labels).sum().item()
        total_sample_num += batch_size

    avg_loss_value = total_loss_value / total_sample_num
    avg_acc_value = total_correct_num / total_sample_num
    return avg_loss_value, avg_acc_value


# 7. Training loop
num_epochs = 10

print("Experiment 3: concat(BERT(context), BERT(utterance)) + MLP")

for epoch_num in range(1, num_epochs + 1):
    train_loss, train_acc = run_one_epoch(
        model=model,
        data_loader=train_loader,
        criterion=criterion,
        device=device,
        optimizer=optimizer
    )

    test_loss, test_acc = run_one_epoch(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device,
        optimizer=None
    )

    print(
        f"Epoch {epoch_num:02d} | "
        f"train loss: {train_loss:.4f} | "
        f"train acc: {train_acc:.4f} | "
        f"test loss: {test_loss:.4f} | "
        f"test acc: {test_acc:.4f}"
    )

Experiment 3: concat(BERT(context), BERT(utterance)) + MLP
Epoch 01 | train loss: 0.6976 | train acc: 0.4946 | test loss: 0.6824 | test acc: 0.6087
Epoch 02 | train loss: 0.6897 | train acc: 0.5145 | test loss: 0.6786 | test acc: 0.6449
Epoch 03 | train loss: 0.6840 | train acc: 0.6014 | test loss: 0.6772 | test acc: 0.6812
Epoch 04 | train loss: 0.6745 | train acc: 0.6286 | test loss: 0.6704 | test acc: 0.7246
Epoch 05 | train loss: 0.6615 | train acc: 0.6884 | test loss: 0.6576 | test acc: 0.7609
Epoch 06 | train loss: 0.6492 | train acc: 0.7138 | test loss: 0.6460 | test acc: 0.7536
Epoch 07 | train loss: 0.6322 | train acc: 0.7210 | test loss: 0.6320 | test acc: 0.7609
Epoch 08 | train loss: 0.6105 | train acc: 0.7464 | test loss: 0.6152 | test acc: 0.7681
Epoch 09 | train loss: 0.5889 | train acc: 0.7826 | test loss: 0.5999 | test acc: 0.7391
Epoch 10 | train loss: 0.5628 | train acc: 0.7989 | test loss: 0.5837 | test acc: 0.7174
